# TrustRAG — Calibration & Selective-Prediction Plots

The two 'money plots' for the README: the **risk-coverage curve** (+ AURC) and the
**reliability diagram** (+ ECE, before vs after calibration).

Run `python scripts/evaluate.py --preds preds/test.jsonl` first to produce
`artifacts/plot_data.json`. To preview the *shape* without real data, run
`python scripts/verify_core.py` logic — or just execute the synthetic cell below.

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt

PLOT_DATA = '../artifacts/plot_data.json'
if os.path.exists(PLOT_DATA):
    data = json.load(open(PLOT_DATA))
    coverage = np.array(data['coverage']); risk = np.array(data['risk'])
    aurc = data['aurc']; reliability = data['reliability']
    print('loaded real eval output:', data.get('b3'))
else:
    # Fallback: synthetic complementary-signal data via the production code path.
    import sys; sys.path.insert(0, '..')
    from scripts.verify_core import make_split
    from trustrag.abstain.gate import train_gate
    from eval import metrics as M
    rng = np.random.default_rng(42)
    Xf,yf = make_split(1000,rng); Xc,yc = make_split(500,rng); Xt,yt = make_split(800,rng)
    gate,_ = train_gate(Xf,yf,Xc,yc)
    conf = gate.predict_confidence_batch(Xt)
    rc = M.risk_coverage_curve(conf, yt)
    coverage, risk, aurc = rc.coverage, rc.risk, rc.aurc
    reliability = M.reliability_bins(conf, yt)
    print('using synthetic fallback (run scripts/evaluate.py for real numbers)')

In [ ]:
# Risk-coverage curve
plt.figure(figsize=(6,4))
plt.plot(coverage, risk, lw=2)
plt.xlabel('coverage (fraction answered)'); plt.ylabel('risk (error among answered)')
plt.title(f'Risk-Coverage Curve  (AURC = {aurc:.3f})')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Reliability diagram
conf_bins = [b[0] for b in reliability]; acc_bins = [b[1] for b in reliability]
plt.figure(figsize=(5,5))
plt.plot([0,1],[0,1],'--',color='gray',label='perfect calibration')
plt.plot(conf_bins, acc_bins, 'o-', lw=2, label='TrustRAG (calibrated)')
plt.xlabel('mean predicted confidence'); plt.ylabel('empirical accuracy')
plt.title('Reliability Diagram'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()